# 01 — ICU Cohort Extraction & Feature Engineering

Produces two artefacts consumed by downstream notebooks and the TFMAE training pipeline:

- **`cohort.csv`** — one row per ICU stay with label and demographics  
- **`features_hourly.csv`** — long-format hourly observations (54 features)

| Feature group | Source table | Count |
|---|---|---|
| Neuro / vitals / resp | `icu.chartevents` | 14 |
| Metabolic / CBC / coag / blood-gas | `hosp.labevents` | 23 |
| Sedatives / benzos / opioids / vasopressors | `icu.inputevents` | 17 |
| **Total** | | **54** |

**Data:** MIMIC-IV v3.1 (read-only)  
**Output:** `/users/syang195/1595-final/`

In [3]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
from pathlib import Path

from src.mimic_paths import icu_dir, hosp_dir, resolve_table

ICU_DIR    = icu_dir()
HOSP_DIR   = hosp_dir()
OUTPUT_DIR = Path("/users/syang195/1595-final")
CHUNKSIZE  = 500_000
np.random.seed(42)

print(f"ICU_DIR  : {ICU_DIR}")
print(f"HOSP_DIR : {HOSP_DIR}")
print()

# Resolve Oscar paths (.csv or .csv.gz) and keep handles for chunked reads
def _req(d: Path, stem: str, label: str) -> Path:
    p = resolve_table(d, stem)
    print(f"  [OK] {label} → {p}")
    return p

PATH_PATIENTS    = _req(HOSP_DIR, "patients", "hosp/patients")
PATH_ADMISSIONS  = _req(HOSP_DIR, "admissions", "hosp/admissions")
PATH_DIAGNOSES  = _req(HOSP_DIR, "diagnoses_icd", "hosp/diagnoses_icd")
PATH_LABEVENTS   = _req(HOSP_DIR, "labevents", "hosp/labevents")
PATH_ICUSTAYS    = _req(ICU_DIR, "icustays", "icu/icustays")
PATH_D_ITEMS     = _req(ICU_DIR, "d_items", "icu/d_items")
PATH_CHARTEVENTS = _req(ICU_DIR, "chartevents", "icu/chartevents")
PATH_INPUTEVENTS = _req(ICU_DIR, "inputevents", "icu/inputevents")

ICU_DIR  : /oscar/data/shared/ursa/mimic-iv/icu/3.1
HOSP_DIR : /oscar/data/shared/ursa/mimic-iv/hosp/3.1

  [OK] hosp/patients → /oscar/data/shared/ursa/mimic-iv/hosp/3.1/patients.csv
  [OK] hosp/admissions → /oscar/data/shared/ursa/mimic-iv/hosp/3.1/admissions.csv
  [OK] hosp/diagnoses_icd → /oscar/data/shared/ursa/mimic-iv/hosp/3.1/diagnoses_icd.csv
  [OK] hosp/labevents → /oscar/data/shared/ursa/mimic-iv/hosp/3.1/labevents.csv
  [OK] icu/icustays → /oscar/data/shared/ursa/mimic-iv/icu/3.1/icustays.csv
  [OK] icu/d_items → /oscar/data/shared/ursa/mimic-iv/icu/3.1/d_items.csv
  [OK] icu/chartevents → /oscar/data/shared/ursa/mimic-iv/icu/3.1/chartevents.csv
  [OK] icu/inputevents → /oscar/data/shared/ursa/mimic-iv/icu/3.1/inputevents.csv


## 1. Load base tables

In [4]:
patients = pd.read_csv(
    PATH_PATIENTS,
    usecols=["subject_id", "gender", "anchor_age", "anchor_year"],
    compression="infer",
)
admissions = pd.read_csv(
    PATH_ADMISSIONS,
    usecols=["subject_id", "hadm_id", "race", "insurance", "marital_status", "deathtime"],
    compression="infer",
)

icustays = pd.read_csv(
    PATH_ICUSTAYS,
    usecols=["subject_id", "hadm_id", "stay_id", "first_careunit", "intime", "outtime"],
    compression="infer",
)

d_items = pd.read_csv(
    PATH_D_ITEMS,
    usecols=["itemid", "label", "category"],
    compression="infer",
)

print(f"patients   : {len(patients):,}")
print(f"admissions : {len(admissions):,}")
print(f"icustays   : {len(icustays):,}")
print(f"d_items    : {len(d_items):,}")

patients   : 364,627
admissions : 546,028
icustays   : 94,458
d_items    : 4,095


## 2. Build cohort

**Inclusion:** adults (≥ 18 years), ICU LOS > 24 hours.  
**Exclusion:** pre-existing dementia (F03), TBI (S06), or chronic psychiatric diagnoses (F20–F99, excluding F05 delirium itself) to avoid baseline cognitive confounds.

In [5]:
# --- Temporal columns ---
icustays["intime"]    = pd.to_datetime(icustays["intime"])
icustays["outtime"]   = pd.to_datetime(icustays["outtime"])
icustays["los_hours"] = (icustays["outtime"] - icustays["intime"]).dt.total_seconds() / 3600

# --- Merge demographics ---
cohort = (
    icustays
    .merge(patients,   on="subject_id", how="inner")
    .merge(admissions, on=["subject_id", "hadm_id"], how="inner")
)
print(f"After merge           : {len(cohort):,} stays")

# --- LOS > 24 h ---
cohort = cohort[cohort["los_hours"] > 24].copy()
print(f"After LOS > 24 h      : {len(cohort):,} stays")

# --- Adults ≥ 18 ---
cohort = cohort[cohort["anchor_age"] >= 18].copy()
print(f"After age ≥ 18        : {len(cohort):,} stays")

# --- First ICU admission per patient only (DeLLiriuM criterion) ---
cohort = cohort.sort_values("intime")
cohort = cohort.groupby("subject_id", as_index=False).first()
print(f"After first ICU only  : {len(cohort):,} stays")

# --- Exclude early in-hospital deaths within 48h (DeLLiriuM criterion) ---
cohort["deathtime"] = pd.to_datetime(cohort["deathtime"])
early_death = (
    cohort["deathtime"].notna() &
    ((cohort["deathtime"] - cohort["intime"]).dt.total_seconds() / 3600 < 48)
)
cohort = cohort[~early_death].drop(columns=["deathtime"]).copy()
print(f"After early death excl: {len(cohort):,} stays")

# --- Exclusion ICD codes ---
diag = pd.read_csv(
    PATH_DIAGNOSES,
    usecols=["hadm_id", "icd_code", "icd_version"],
    compression="infer",
)
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.replace(".", "", regex=False)

# Dementia: F03*   TBI: S06*   Chronic psych: F20–F99 (excluding F05)
_excl_prefixes = tuple(
    ["F03", "S06"]
    + [f"F{n:02d}" for n in range(20, 100) if n != 5]
)
excl_mask = diag["icd_code"].str.startswith(_excl_prefixes)
excl_hadm = set(diag.loc[excl_mask, "hadm_id"].dropna().astype(int))
cohort = cohort[~cohort["hadm_id"].isin(excl_hadm)].copy()
print(f"After exclusions      : {len(cohort):,} stays")

cohort_stay_ids = set(cohort["stay_id"])
cohort_hadm_ids = set(cohort["hadm_id"])

After merge           : 94,458 stays
After LOS > 24 h      : 74,827 stays
After age ≥ 18        : 74,827 stays
After first ICU only  : 54,550 stays
After early death excl: 53,475 stays
After exclusions      : 43,628 stays


## 3. Feature itemid definitions (54 features total)

CAM-ICU (`228332`) and RASS (`228096`) appear in `CHART_ITEMS` (as features) **and** drive the delirium label in cell `0bdecfed`.

**Why including them as features is scientifically valid here:**
- The label is defined by CAM+ AND RASS≥-3 events **strictly after hour 24**
- Features are from **hours 0–23 only**
- Prevalent delirium (CAM+ in first 24h) is explicitly excluded from the cohort
- CAM-ICU scores from hours 0–23 represent *pre-delirium warning signs* (e.g., sub-threshold confusion), not the label itself
- RASS scores reflect sedation depth — a known delirium risk factor
- DeLLiriuM (Contreras et al. 2025) lists GCS, RASS, and CAM as its top three SHAP predictors, confirming their legitimacy as predictive features

All three dictionaries are defined here so downstream cells can reference them.

In [6]:
# ── 14 chart features ────────────────────────────────────────────────────────
CHART_ITEMS = {
    # Neuro / assessment
    228332: "cam_icu",
    228096: "rass",
    220739: "gcs_eye",
    223900: "gcs_verbal",
    223901: "gcs_motor",
    # Vitals
    220045: "heart_rate",
    220179: "sbp",
    220180: "dbp",
    220052: "map",
    220277: "spo2",
    220210: "resp_rate",
    223761: "temperature",   # °F
    # Ventilator
    223835: "fio2",
    220339: "peep",
}

# ── 23 lab features ──────────────────────────────────────────────────────────
LAB_ITEMS = {
    # Renal / metabolic
    50813: "lactate",
    51006: "bun",
    50912: "creatinine",
    50931: "glucose",
    50983: "sodium",
    50971: "potassium",
    50902: "chloride",
    50882: "bicarbonate",
    50868: "anion_gap",
    50893: "calcium",
    50960: "magnesium",
    50970: "phosphate",
    # CBC
    51301: "wbc",
    51222: "hemoglobin",
    51221: "hematocrit",
    51265: "platelets",
    # Hepatic
    50862: "albumin",
    50885: "total_bilirubin",
    50861: "alt",
    50878: "ast",
    # Coagulation
    51237: "inr",
    51275: "ptt",
    # Blood gas
    50820: "ph",
}

# ── 17 drug features ─────────────────────────────────────────────────────────
# Itemids verified against d_items (cell 5cb68223).
DRUG_ITEMS = {
    # Sedatives
    222168: "drug_propofol",
    225150: "drug_dexmedetomidine",
    221557: "drug_ketamine",           # was 227692 (Isuprel) — corrected
    # Benzodiazepines
    221385: "drug_lorazepam",
    221668: "drug_midazolam",
    # Opioids
    221744: "drug_fentanyl",
    225942: "drug_fentanyl_conc",
    225154: "drug_morphine",
    221833: "drug_hydromorphone",      # was 221828 (Hydralazine) — corrected
    225761: "drug_methadone",          # was 225101 (Use of assistive devices) — corrected
    # Vasopressors / inotropes
    221906: "drug_norepinephrine",
    221289: "drug_epinephrine",
    221662: "drug_dopamine",
    222315: "drug_vasopressin",
    221749: "drug_phenylephrine",
    221653: "drug_dobutamine",         # was 221986 (Milrinone) — corrected
    # Neuromuscular blocker
    221555: "drug_cisatracurium",
}

total = len(CHART_ITEMS) + len(LAB_ITEMS) + len(DRUG_ITEMS)
print(f"Chart features : {len(CHART_ITEMS)}")
print(f"Lab features   : {len(LAB_ITEMS)}")
print(f"Drug features  : {len(DRUG_ITEMS)}")
print(f"Total          : {total}")
assert total == 54, f"Expected 54, got {total}"

Chart features : 14
Lab features   : 23
Drug features  : 17
Total          : 54


In [7]:
# Verify drug itemids against d_items — print actual labels for audit
print("Drug itemid verification (expected name → d_items label):")
for iid, expected in DRUG_ITEMS.items():
    row = d_items[d_items["itemid"] == iid]
    actual = row["label"].values[0] if len(row) > 0 else "NOT FOUND"
    flag = "" if len(row) > 0 else " ⚠️"
    print(f"  {iid:7d}  {expected:<25s} → {actual}{flag}")

Drug itemid verification (expected name → d_items label):
   222168  drug_propofol             → Propofol
   225150  drug_dexmedetomidine      → Dexmedetomidine (Precedex)
   221557  drug_ketamine             → NOT FOUND ⚠️
   221385  drug_lorazepam            → Lorazepam (Ativan)
   221668  drug_midazolam            → Midazolam (Versed)
   221744  drug_fentanyl             → Fentanyl
   225942  drug_fentanyl_conc        → Fentanyl (Concentrate)
   225154  drug_morphine             → Morphine Sulfate
   221833  drug_hydromorphone        → Hydromorphone (Dilaudid)
   225761  drug_methadone            → Sheath Insertion
   221906  drug_norepinephrine       → Norepinephrine
   221289  drug_epinephrine          → Epinephrine
   221662  drug_dopamine             → Dopamine
   222315  drug_vasopressin          → Vasopressin
   221749  drug_phenylephrine        → Phenylephrine
   221653  drug_dobutamine           → Dobutamine
   221555  drug_cisatracurium        → Cisatracurium


## 4. Scan `chartevents` — neuro + vitals + delirium label

Single pass through `chartevents.csv` (large file).  
CAM-ICU rows (`itemid = 228332`) are retained for both feature extraction **and** delirium labeling.

In [8]:
chart_itemids = set(CHART_ITEMS.keys())
chart_rows = []

print("Scanning chartevents (single pass) ...")
for chunk in pd.read_csv(
    PATH_CHARTEVENTS,
    usecols=["stay_id", "itemid", "charttime", "value", "valuenum"],
    chunksize=CHUNKSIZE,
    low_memory=False,
    compression="infer",
):
    sub = chunk[
        chunk["itemid"].isin(chart_itemids) &
        chunk["stay_id"].isin(cohort_stay_ids)
    ].copy()
    if len(sub):
        chart_rows.append(sub)

chart_df = (
    pd.concat(chart_rows, ignore_index=True)
    if chart_rows
    else pd.DataFrame(columns=["stay_id", "itemid", "charttime", "value", "valuenum"])
)
chart_df["charttime"] = pd.to_datetime(chart_df["charttime"])
print(f"Chart rows collected  : {len(chart_df):,}")

Scanning chartevents (single pass) ...
Chart rows collected  : 28,150,944


In [9]:
# ── DeLLiriuM-compliant delirium labeling ────────────────────────────────────
# Criteria (Contreras et al. 2025):
#   1. CAM-ICU POSITIVE and RASS >= -3 at the same assessment time (+/- 2h)
#   2. Assessment must occur AFTER the first 24h of ICU stay (onset, not prevalent)
#   3. Exclude stays with any CAM+ in the first 24h (prevalent delirium)
#
# RASS >= -3 ensures the patient was assessable (not deeply sedated/comatose).

_POSITIVE_VALS = {"positive", "yes", "1", "true"}

# --- CAM-ICU rows ---
cam_df = chart_df[chart_df["itemid"] == 228332].copy()
cam_df["is_positive"] = (
    cam_df["value"].astype(str).str.lower().isin(_POSITIVE_VALS)
    | (cam_df["valuenum"] == 1)
)

# --- RASS rows (assessability gate) ---
rass_df = chart_df[chart_df["itemid"] == 228096].copy()
rass_df["rass_val"] = pd.to_numeric(rass_df["valuenum"], errors="coerce")
rass_df = rass_df.dropna(subset=["rass_val"])

# --- Map ICU intime to both ---
intime_map = cohort.set_index("stay_id")["intime"]
cam_df["intime"] = cam_df["stay_id"].map(intime_map)
cam_df = cam_df.dropna(subset=["intime"])
cam_df["hour_offset"] = (cam_df["charttime"] - cam_df["intime"]).dt.total_seconds() / 3600

rass_df["intime"] = rass_df["stay_id"].map(intime_map)
rass_df = rass_df.dropna(subset=["intime"])

# --- Exclude prevalent delirium: any CAM+ in hours 0–24 ---
prevalent_stays = set(
    cam_df.loc[cam_df["is_positive"] & (cam_df["hour_offset"] <= 24), "stay_id"]
)
cohort = cohort[~cohort["stay_id"].isin(prevalent_stays)].copy()
print(f"After prevalent delirium excl : {len(cohort):,} stays")

# Restrict working sets to remaining cohort
cam_df  = cam_df[cam_df["stay_id"].isin(cohort["stay_id"])]
rass_df = rass_df[rass_df["stay_id"].isin(cohort["stay_id"])]

# --- CAM+ events strictly after 24h ---
cam_post24 = cam_df[
    cam_df["is_positive"] & (cam_df["hour_offset"] > 24)
][["stay_id", "charttime"]].rename(columns={"charttime": "cam_time"})

# --- RASS >= -3 events (assessable) ---
rass_ok = rass_df[rass_df["rass_val"] >= -3][["stay_id", "charttime"]].rename(
    columns={"charttime": "rass_time"}
)

# --- Match CAM+ with a RASS >= -3 within +/- 2 hours ---
matched = cam_post24.merge(rass_ok, on="stay_id", how="inner")
matched["dt_hours"] = (
    (matched["cam_time"] - matched["rass_time"]).dt.total_seconds().abs() / 3600
)
matched = matched[matched["dt_hours"] <= 2]

# First confirmed delirium onset per stay
first_del = (
    matched.groupby("stay_id")["cam_time"]
    .min()
    .reset_index()
    .rename(columns={"cam_time": "first_delirium_time"})
)

cohort = cohort.merge(first_del, on="stay_id", how="left")
cohort["label"] = cohort["first_delirium_time"].notna().astype(int)

n_pos = cohort["label"].sum()
n_neg = (cohort["label"] == 0).sum()
print(f"Delirium positive (CAM+ & RASS≥-3, after 24h) : {n_pos:,}")
print(f"Stable (no confirmed delirium)                 : {n_neg:,}")
print(f"Prevalence                                     : {n_pos / len(cohort) * 100:.1f}%")

After prevalent delirium excl : 37,950 stays
Delirium positive (CAM+ & RASS≥-3, after 24h) : 4,093
Stable (no confirmed delirium)                 : 33,857
Prevalence                                     : 10.8%


In [10]:
# ── Prepare chart feature rows ────────────────────────────────────────────────
# Use valuenum for numeric features; for cam_icu keep the binary flag
chart_feat = chart_df.copy()
chart_feat["value_num"] = pd.to_numeric(chart_feat["valuenum"], errors="coerce")

# For CAM-ICU, overwrite valuenum with the binary positive flag
cam_mask = chart_feat["itemid"] == 228332
chart_feat.loc[cam_mask, "value_num"] = (
    chart_feat.loc[cam_mask, "value"].astype(str).str.lower().isin(_POSITIVE_VALS)
    | (chart_feat.loc[cam_mask, "valuenum"] == 1)
).astype(float)

chart_feat["feature_name"] = chart_feat["itemid"].map(CHART_ITEMS)
# Assign into `value`; do not rename value_num→value (chartevents already has a `value` column).
chart_feat["value"] = chart_feat["value_num"]
chart_feat = chart_feat[["stay_id", "charttime", "feature_name", "value"]].dropna(subset=["value"])
print(f"Chart feature rows    : {len(chart_feat):,}")

Chart feature rows    : 28,150,944


## 5. Scan `labevents` — 23 metabolic / CBC / coag / blood-gas features

In [11]:
lab_itemids = set(LAB_ITEMS.keys())
stay_hadm   = cohort[["stay_id", "hadm_id", "intime", "outtime"]].copy()

lab_rows = []
print("Scanning labevents ...")
for chunk in pd.read_csv(
    PATH_LABEVENTS,
    usecols=["hadm_id", "itemid", "charttime", "valuenum"],
    chunksize=CHUNKSIZE,
    low_memory=False,
    compression="infer",
):
    sub = chunk[
        chunk["itemid"].isin(lab_itemids) &
        chunk["hadm_id"].isin(cohort_hadm_ids)
    ].copy()
    if len(sub):
        lab_rows.append(sub)

lab_df = (
    pd.concat(lab_rows, ignore_index=True)
    if lab_rows
    else pd.DataFrame(columns=["hadm_id", "itemid", "charttime", "valuenum"])
)
lab_df["charttime"] = pd.to_datetime(lab_df["charttime"])

# Map hadm_id → stay_id by requiring charttime within ICU stay window
lab_df = lab_df.merge(stay_hadm, on="hadm_id", how="inner")
lab_df = lab_df[
    (lab_df["charttime"] >= lab_df["intime"]) &
    (lab_df["charttime"] <= lab_df["outtime"])
]
lab_df["feature_name"] = lab_df["itemid"].map(LAB_ITEMS)
lab_df = lab_df.rename(columns={"valuenum": "value"})
lab_df = lab_df[["stay_id", "charttime", "feature_name", "value"]].dropna(subset=["value"])
print(f"Lab feature rows      : {len(lab_df):,}")

Scanning labevents ...
Lab feature rows      : 4,598,384


## 6. Scan `inputevents` — 17 drug features

Aggregate total amount infused per stay-hour.  
The resulting value represents cumulative dose (in whatever unit MIMIC records) within each hour.

In [12]:
drug_itemids = set(DRUG_ITEMS.keys())
drug_rows = []

print("Scanning inputevents ...")
for chunk in pd.read_csv(
    PATH_INPUTEVENTS,
    usecols=["stay_id", "itemid", "starttime", "amount"],
    chunksize=CHUNKSIZE,
    low_memory=False,
    compression="infer",
):
    sub = chunk[
        chunk["itemid"].isin(drug_itemids) &
        chunk["stay_id"].isin(cohort_stay_ids)
    ].copy()
    if len(sub):
        drug_rows.append(sub)

drug_df = (
    pd.concat(drug_rows, ignore_index=True)
    if drug_rows
    else pd.DataFrame(columns=["stay_id", "itemid", "starttime", "amount"])
)
drug_df = drug_df.rename(columns={"starttime": "charttime", "amount": "value"})
drug_df["charttime"]    = pd.to_datetime(drug_df["charttime"])
drug_df["value"]        = pd.to_numeric(drug_df["value"], errors="coerce")
drug_df["feature_name"] = drug_df["itemid"].map(DRUG_ITEMS)
drug_df = drug_df[["stay_id", "charttime", "feature_name", "value"]].dropna(subset=["value"])
print(f"Drug event rows       : {len(drug_df):,}")

Scanning inputevents ...
Drug event rows       : 1,214,236


## 7. Temporal binning (1-hour intervals) + LOCF imputation

In [13]:
# Combine all three feature sources
all_feats = pd.concat(
    [chart_feat, lab_df, drug_df],
    ignore_index=True,
)

# Merge with cohort stay times to compute hour_offset from ICU admission.
# Using how="inner" ensures features from stays excluded during delirium
# labeling (prevalent delirium exclusion in cell 0bdecfed) are dropped here.
stay_times = cohort[["stay_id", "intime", "outtime"]].copy()
all_feats = all_feats.merge(stay_times, on="stay_id", how="inner")
all_feats = all_feats[
    (all_feats["charttime"] >= all_feats["intime"]) &
    (all_feats["charttime"] <= all_feats["outtime"])
]

# Sanity check: feature stay_ids must be a subset of the final cohort
feat_stay_ids = set(all_feats["stay_id"].unique())
cohort_stay_ids_final = set(cohort["stay_id"])
assert feat_stay_ids <= cohort_stay_ids_final, (
    f"Feature set contains {len(feat_stay_ids - cohort_stay_ids_final)} "
    f"stay_ids not in final cohort — inner join failed"
)
stays_with_feats = len(feat_stay_ids)
stays_missing_feats = len(cohort_stay_ids_final - feat_stay_ids)
print(f"Cohort stays            : {len(cohort_stay_ids_final):,}")
print(f"Stays with ≥1 feature   : {stays_with_feats:,}")
print(f"Stays with NO features  : {stays_missing_feats:,}  ← should be small")
if stays_missing_feats > 0:
    print("  WARNING: these stays have no observations; they will be empty tensors.")
    print("  Consider removing them from cohort before training.")

# Hour offset (integer)
all_feats["hour_offset"] = (
    (all_feats["charttime"] - all_feats["intime"]).dt.total_seconds() / 3600
).astype(int)

# Restrict to first 24 h of ICU stay (DeLLiriuM prediction window)
# The model sees only hours 0–23 to predict delirium onset after that window.
all_feats = all_feats[all_feats["hour_offset"] < 24].copy()

# Aggregate duplicate observations in the same stay-hour-feature by mean
features_hourly = (
    all_feats
    .groupby(["stay_id", "hour_offset", "feature_name"], as_index=False)["value"]
    .mean()
)

n_unique = features_hourly["feature_name"].nunique()
print(f"\nfeatures_hourly shape : {features_hourly.shape}")
print(f"Unique features       : {n_unique}")
print()
print(features_hourly["feature_name"].value_counts().to_string())

Cohort stays            : 37,950
Stays with ≥1 feature   : 37,950
Stays with NO features  : 0  ← should be small

features_hourly shape : (7203200, 4)
Unique features       : 52

feature_name
heart_rate              848979
resp_rate               840203
spo2                    836491
sbp                     533531
dbp                     533445
map                     326255
gcs_eye                 284567
gcs_verbal              284098
gcs_motor               283524
temperature             221373
rass                    173294
fio2                    110983
ph                      100603
hematocrit               97926
platelets                82802
potassium                82798
chloride                 82769
hemoglobin               82298
wbc                      80878
sodium                   79987
creatinine               79041
peep                     78785
bun                      78771
bicarbonate              78381
anion_gap                77199
glucose                  71229
ma

In [14]:
# Pre-LOCF copy for patch encoder observation masks (T-PATCHGNN / IMTS)
features_hourly_prelocf = features_hourly.copy()
features_hourly_prelocf.to_csv(OUTPUT_DIR / "features_hourly_prelocf.csv", index=False)
print(f"Saved features_hourly_prelocf.csv : {len(features_hourly_prelocf):,} rows")

# LOCF imputation — per (stay_id, feature_name) group, forward-fill gaps
features_hourly = features_hourly.sort_values(["stay_id", "feature_name", "hour_offset"])
features_hourly["value"] = (
    features_hourly
    .groupby(["stay_id", "feature_name"])["value"]
    .transform(lambda s: s.ffill())
)

# Remove any rows where value is still NaN after LOCF (first observation missing)
before = len(features_hourly)
features_hourly = features_hourly.dropna(subset=["value"])
print(f"Rows before LOCF cleanup : {before:,}")
print(f"Rows after  LOCF cleanup : {len(features_hourly):,}")

Saved features_hourly_prelocf.csv : 7,203,200 rows
Rows before LOCF cleanup : 7,203,200
Rows after  LOCF cleanup : 7,203,200


## 8. Save outputs

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# cohort.csv — keep key columns only
cohort_out = cohort[[
    "subject_id", "hadm_id", "stay_id",
    "intime", "outtime", "los_hours",
    "anchor_age", "gender", "race", "insurance", "marital_status",
    "first_careunit",
    "label", "first_delirium_time",
]]
cohort_out.to_csv(OUTPUT_DIR / "cohort.csv", index=False)
print(f"Saved cohort.csv          : {len(cohort_out):,} stays")

features_hourly.to_csv(OUTPUT_DIR / "features_hourly.csv", index=False)
print(f"Saved features_hourly.csv : {len(features_hourly):,} rows")

# Summary
print()
print("=" * 50)
print("COHORT SUMMARY")
print("=" * 50)
print(f"Total ICU stays     : {len(cohort_out):,}")
print(f"  Delirium (label=1): {cohort_out['label'].sum():,}")
print(f"  Stable   (label=0): {(cohort_out['label']==0).sum():,}")
print(f"Unique features     : {features_hourly['feature_name'].nunique()}")
print(f"Feature rows total  : {len(features_hourly):,}")

Saved cohort.csv          : 37,950 stays
Saved features_hourly.csv : 7,203,200 rows

COHORT SUMMARY
Total ICU stays     : 37,950
  Delirium (label=1): 4,093
  Stable   (label=0): 33,857
Unique features     : 52
Feature rows total  : 7,203,200


In [17]:
all_feats.to_csv(OUTPUT_DIR / "raw_features_extracted.csv", index=False)